# Pruning

In [1]:
from model import Classifier

model = Classifier.load_from_checkpoint(
    "lightning_logs/mnist_cnn/version_1/checkpoints/epoch=4-step=8595.ckpt"
)

model.eval()

Classifier(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Linear(in_features=3136, out_features=10, bias=True)
)

In [3]:
import torch.nn.utils.prune as prune

prune.l1_unstructured(
    model.classifier,
    name="weight",
    amount=0.4
)

print(model.classifier.weight.data)

tensor([[-0.0000, -0.0000,  0.0000,  ..., -0.2411, -0.2884, -0.0000],
        [ 0.0000,  0.0000, -0.0588,  ..., -0.1757,  0.0564, -0.1041],
        [-0.0000,  0.0000,  0.1031,  ...,  0.1476, -0.1042, -0.0405],
        ...,
        [-0.0000, -0.0000, -0.0978,  ...,  0.1117,  0.1303, -0.0730],
        [-0.0000, -0.0000, -0.0000,  ..., -0.1026, -0.1081,  0.0808],
        [-0.0000, -0.0000, -0.0000,  ...,  0.0769,  0.1841, -0.0385]])


In [4]:
prune.remove(
    model.classifier,
    'weight'
)

print(model)

Classifier(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Linear(in_features=3136, out_features=10, bias=True)
)


In [5]:
from torchinfo import summary

summary(
    model,
    input_size=(1, 1, 28, 28)
)

Layer (type:depth-idx)                   Output Shape              Param #
Classifier                               [1, 10]                   --
├─Sequential: 1-1                        [1, 64, 7, 7]             --
│    └─Conv2d: 2-1                       [1, 32, 28, 28]           320
│    └─ReLU: 2-2                         [1, 32, 28, 28]           --
│    └─MaxPool2d: 2-3                    [1, 32, 14, 14]           --
│    └─Conv2d: 2-4                       [1, 64, 14, 14]           18,496
│    └─ReLU: 2-5                         [1, 64, 14, 14]           --
│    └─MaxPool2d: 2-6                    [1, 64, 7, 7]             --
├─Linear: 1-2                            [1, 10]                   31,370
Total params: 50,186
Trainable params: 50,186
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 3.91
Input size (MB): 0.00
Forward/backward pass size (MB): 0.30
Params size (MB): 0.20
Estimated Total Size (MB): 0.51

In [7]:
import torch
def count_zero_weights(model):
    total = 0
    zeros = 0

    for name, param in model.named_parameters():
        total += param.numel()
        zeros += torch.sum(param == 0).item()

    return zeros, total


zeros, total = count_zero_weights(model)

print(f"Zero weights: {zeros}/{total}")
print(f"Sparsity: {100 * zeros / total:.2f}%")

Zero weights: 12544/50186
Sparsity: 25.00%
